In [1]:
from datasets import load_dataset

DATASET_ID = "lmsys/mt_bench_human_judgments"
dataset = load_dataset(DATASET_ID)

dataset

/home/snt/projects_lujun/LLMJudgeTempCausal/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    gpt4_pair: Dataset({
        features: ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn'],
        num_rows: 2400
    })
    human: Dataset({
        features: ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn'],
        num_rows: 3355
    })
})

In [2]:
def run_dataset_smoke_tests(dataset_dict):
    assert len(dataset_dict) > 0, "Dataset has no splits"

    report = []
    for split_name, split in dataset_dict.items():
        assert split.num_rows > 0, f"Split '{split_name}' is empty"
        assert len(split.column_names) > 0, f"Split '{split_name}' has no columns"

        first_row = split[0]
        assert isinstance(first_row, dict), f"Split '{split_name}' row is not a dict"
        assert set(first_row.keys()) == set(split.column_names), (
            f"Split '{split_name}' row keys do not match column names"
        )

        report.append(
            {
                "split": split_name,
                "rows": split.num_rows,
                "columns": split.column_names,
            }
        )

    return report

smoke_test_report = run_dataset_smoke_tests(dataset)
smoke_test_report

[{'split': 'gpt4_pair',
  'rows': 2400,
  'columns': ['question_id',
   'model_a',
   'model_b',
   'winner',
   'judge',
   'conversation_a',
   'conversation_b',
   'turn']},
 {'split': 'human',
  'rows': 3355,
  'columns': ['question_id',
   'model_a',
   'model_b',
   'winner',
   'judge',
   'conversation_a',
   'conversation_b',
   'turn']}]

In [8]:
from pprint import pprint

split_name = "human" # gpt4_pair
split = dataset[split_name]

print(f"Using split: {split_name}")
print(f"Rows: {split.num_rows}")
print(f"Columns ({len(split.column_names)}): {split.column_names}")

print("\nExample row:")
pprint(split[0])

print("\nFirst 5 rows as pandas DataFrame:")
display(split.select(range(min(5, split.num_rows))).to_pandas())

Using split: human
Rows: 3355
Columns (8): ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn']

Example row:
{'conversation_a': [{'content': 'Compose an engaging travel blog post about a '
                                'recent trip to Hawaii, highlighting cultural '
                                'experiences and must-see attractions.',
                     'role': 'user'},
                    {'content': 'I recently had the pleasure of visiting '
                                'Hawaii and it quickly became one of my '
                                'favorite places. From the stunning beaches to '
                                'the lush mountains, this place has it all. '
                                'The people are incredibly friendly and the '
                                'culture is alive and well. One of the '
                                'highlights of my trip was visiting the '
                                'Polyn

,question_id,model_a,model_b,winner,judge,conversation_a,conversation_b,turn
0,81,alpaca-13b,gpt-3.5-turbo,model_b,author_2,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,1
1,81,alpaca-13b,gpt-3.5-turbo,model_b,author_2,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,2
2,81,alpaca-13b,gpt-3.5-turbo,model_b,expert_17,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,1
3,81,alpaca-13b,gpt-3.5-turbo,model_b,expert_17,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,2
4,81,alpaca-13b,vicuna-13b-v1.2,model_b,expert_0,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,1


In [9]:
sample_n = 350

# 随机采样 350 条（若不足 350 条则取全部）
sample_size = min(sample_n, split.num_rows)
sampled_split = split.shuffle(seed=42).select(range(sample_size))

print(f"Sampled {sample_size} rows from split '{split_name}'")
sampled_df = sampled_split.to_pandas()
display(sampled_df.head(10))

# sampled_df 就是这 350 条数据

Sampled 350 rows from split 'human'


,question_id,model_a,model_b,winner,judge,conversation_a,conversation_b,turn
0,121,gpt-3.5-turbo,gpt-4,model_a,expert_8,[{'content': 'Develop a Python program that re...,[{'content': 'Develop a Python program that re...,2
1,112,llama-13b,vicuna-13b-v1.2,model_a,expert_32,[{'content': 'A tech startup invests $8000 in ...,[{'content': 'A tech startup invests $8000 in ...,2
2,138,vicuna-13b-v1.2,gpt-3.5-turbo,model_b,expert_42,[{'content': 'Analyze the following customer r...,[{'content': 'Analyze the following customer r...,2
3,99,gpt-4,gpt-3.5-turbo,tie,expert_11,[{'content': 'Suppose you are a mathematician ...,[{'content': 'Suppose you are a mathematician ...,1
4,93,vicuna-13b-v1.2,gpt-4,model_b,expert_29,[{'content': 'Imagine yourself as a doctor tas...,[{'content': 'Imagine yourself as a doctor tas...,2
5,133,gpt-3.5-turbo,vicuna-13b-v1.2,tie,author_0,[{'content': 'Extract the following informatio...,[{'content': 'Extract the following informatio...,1
6,141,llama-13b,gpt-4,model_b,author_3,"[{'content': 'In the field of quantum physics,...","[{'content': 'In the field of quantum physics,...",2
7,108,gpt-3.5-turbo,vicuna-13b-v1.2,tie,expert_53,[{'content': 'Which word does not belong with ...,[{'content': 'Which word does not belong with ...,1
8,94,claude-v1,alpaca-13b,model_a,expert_2,[{'content': 'Please take on the role of a rel...,[{'content': 'Please take on the role of a rel...,1
9,152,gpt-3.5-turbo,llama-13b,model_a,expert_57,[{'content': 'How do the stages of life shape ...,[{'content': 'How do the stages of life shape ...,1


In [10]:
sampled_df.judge.value_counts()

judge
author_0     22
expert_24    20
author_4     20
author_2     12
author_3     11
             ..
expert_22     2
expert_38     1
expert_14     1
expert_18     1
expert_52     1
Name: count, Length: 63, dtype: int64

In [12]:
from datasets import load_dataset
import pandas as pd

# 下载数据集（test split）
ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test")

# 转为 DataFrame
df = ds.to_pandas()
print(f"In total {len(df)} 条记录，字段：{list(df.columns)}")
df.head(5)

In total 12032 条记录，字段：['question_id', 'question', 'options', 'answer', 'answer_index', 'cot_content', 'category', 'src']


,question_id,question,options,answer,answer_index,cot_content,category,src
0,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,,business,ori_mmlu-business_ethics
1,71,Managers are entrusted to run the company in t...,"[Shareholders, Diligence, Self-interest, Share...",F,5,,business,ori_mmlu-business_ethics
2,72,There are two main issues associated with ____...,"[Down, Autonomy, Remuneration, Benefit, Down, ...",J,9,,business,ori_mmlu-business_ethics
3,73,_______ locate morality beyond the sphere of r...,"[Ethical egoism, Ethics of duty, Postmodern et...",C,2,,business,ori_mmlu-business_ethics
4,74,Some of key differences between Islamic finan...,"[Interest, Certain, Assured, Both tangible and...",G,6,,business,ori_mmlu-business_ethics


In [13]:
# 按学科统计题目数量
print("各学科题目数：")
display(df["category"].value_counts().rename_axis("category").reset_index(name="count"))

# 看一道题的完整内容
sample = df.iloc[0]
print(f"\n题目：{sample['question']}")
print(f"选项：{sample['options']}")
print(f"正确答案：{sample['answer']} (index {sample['answer_index']})")

各学科题目数：


,category,count
0,math,1351
1,physics,1299
2,chemistry,1132
3,law,1101
4,engineering,969
5,other,924
6,economics,844
7,health,818
8,psychology,798
9,business,789



题目：Typical advertising regulatory bodies suggest, for example that adverts must not: encourage _________, cause unnecessary ________ or _____, and must not cause _______ offence.
选项：['Safe practices, Fear, Jealousy, Trivial'
 'Unsafe practices, Distress, Joy, Trivial'
 'Safe practices, Wants, Jealousy, Trivial'
 'Safe practices, Distress, Fear, Trivial'
 'Unsafe practices, Wants, Jealousy, Serious'
 'Safe practices, Distress, Jealousy, Serious'
 'Safe practices, Wants, Fear, Serious'
 'Unsafe practices, Wants, Fear, Trivial'
 'Unsafe practices, Distress, Fear, Serious']
正确答案：I (index 8)


In [14]:
target_n = 150

# 按 category 比例计算每类应抽多少条
category_counts = df["category"].value_counts().sort_index()
raw_counts = category_counts / len(df) * target_n
sample_counts = raw_counts.astype(int)

# 把余下的名额按小数部分从大到小补齐，确保总数正好是 150
remainder = target_n - sample_counts.sum()
if remainder > 0:
    extra_categories = (raw_counts - sample_counts).sort_values(ascending=False).index[:remainder]
    sample_counts.loc[extra_categories] += 1

# 分类别抽样
sampled_parts = []
for category, n in sample_counts.items():
    part = df[df["category"] == category].sample(n=int(n), random_state=42)
    sampled_parts.append(part)

sampled_df_mmlu_pro = pd.concat(sampled_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"抽样后共 {len(sampled_df_mmlu_pro)} 条")
print("\n抽样后的 category 分布：")
display(sampled_df_mmlu_pro["category"].value_counts().rename_axis("category").reset_index(name="count"))

sampled_df_mmlu_pro.head()

抽样后共 150 条

抽样后的 category 分布：


,category,count
0,math,17
1,physics,16
2,law,14
3,chemistry,14
4,engineering,12
5,other,11
6,economics,11
7,business,10
8,health,10
9,psychology,10


,question_id,question,options,answer,answer_index,cot_content,category,src
0,4715,This question refers to the following informat...,[Logan believes that Indians need to find stre...,F,5,,history,ori_mmlu-high_school_us_history
1,208,You are asked to determine the price of a Euro...,"[16.4, 9.7, 11.9, 15.6, 13.1, 14.2, 7.8, 8.5, ...",C,2,,business,theoremQA-Finance
2,11245,When was the major shift by Greek philosopher...,"[Late Sixth Century BCE, Late Second Century B...",A,0,,philosophy,ori_mmlu-world_religions
3,1286,Which of the following criticisms of Llewellyn...,[There is no distinction between the two forms...,C,2,,law,ori_mmlu-jurisprudence
4,1200,Is the recognition of foreign judgments subjec...,[Foreign judgments are enforced on the basis o...,D,3,,law,ori_mmlu-international_law


In [15]:
sampled_df

,question_id,model_a,model_b,winner,judge,conversation_a,conversation_b,turn
0,121,gpt-3.5-turbo,gpt-4,model_a,expert_8,[{'content': 'Develop a Python program that re...,[{'content': 'Develop a Python program that re...,2
1,112,llama-13b,vicuna-13b-v1.2,model_a,expert_32,[{'content': 'A tech startup invests $8000 in ...,[{'content': 'A tech startup invests $8000 in ...,2
2,138,vicuna-13b-v1.2,gpt-3.5-turbo,model_b,expert_42,[{'content': 'Analyze the following customer r...,[{'content': 'Analyze the following customer r...,2
3,99,gpt-4,gpt-3.5-turbo,tie,expert_11,[{'content': 'Suppose you are a mathematician ...,[{'content': 'Suppose you are a mathematician ...,1
4,93,vicuna-13b-v1.2,gpt-4,model_b,expert_29,[{'content': 'Imagine yourself as a doctor tas...,[{'content': 'Imagine yourself as a doctor tas...,2
...,...,...,...,...,...,...,...,...
345,147,claude-v1,gpt-4,model_b,expert_28,[{'content': 'The city of Vega intends to buil...,[{'content': 'The city of Vega intends to buil...,1
346,93,claude-v1,vicuna-13b-v1.2,model_a,author_5,[{'content': 'Imagine yourself as a doctor tas...,[{'content': 'Imagine yourself as a doctor tas...,1
347,128,gpt-3.5-turbo,vicuna-13b-v1.2,model_a,expert_29,[{'content': 'A binary tree is full if all of ...,[{'content': 'A binary tree is full if all of ...,1
348,134,llama-13b,gpt-4,model_a,expert_51,"[{'content': 'Given the following data, identi...","[{'content': 'Given the following data, identi...",1


In [16]:
sampled_df_mmlu_pro

,question_id,question,options,answer,answer_index,cot_content,category,src
0,4715,This question refers to the following informat...,[Logan believes that Indians need to find stre...,F,5,,history,ori_mmlu-high_school_us_history
1,208,You are asked to determine the price of a Euro...,"[16.4, 9.7, 11.9, 15.6, 13.1, 14.2, 7.8, 8.5, ...",C,2,,business,theoremQA-Finance
2,11245,When was the major shift by Greek philosopher...,"[Late Sixth Century BCE, Late Second Century B...",A,0,,philosophy,ori_mmlu-world_religions
3,1286,Which of the following criticisms of Llewellyn...,[There is no distinction between the two forms...,C,2,,law,ori_mmlu-jurisprudence
4,1200,Is the recognition of foreign judgments subjec...,[Foreign judgments are enforced on the basis o...,D,3,,law,ori_mmlu-international_law
...,...,...,...,...,...,...,...,...
145,4942,This question refers to the following informat...,[Dutch traders forced American ships to extend...,F,5,,history,ori_mmlu-high_school_us_history
146,7996,"7.3-5. In order to estimate the proportion, $p...","[$0.1000$, $0.1600$, $0.1200$, $0.2000$, $0.22...",J,9,,math,scibench-stat
147,287,"An automobile that cost $3,000 four years ago ...","[$300, $2,000, $250, $1,000, $800, $400, $500,...",G,6,,business,stemez-Business
148,7999,Let $A$ and $B$ be independent events with $P(...,"[0.28, 0.9, 0.7, 0.1, 0.56, 0.04, 0.14, 0.5, 0...",G,6,,math,scibench-stat


In [17]:
# 给两个 df 分别加上来源标记
mt_df = sampled_df.copy()
mt_df["dataset"] = "mt_bench_human_judgments"

mmlu_df = sampled_df_mmlu_pro.copy()
mmlu_df["dataset"] = "mmlu_pro"

# 垂直拼接（保留所有列，缺失列自动补 NaN）
combined_df = pd.concat([mt_df, mmlu_df], ignore_index=True, sort=False)

print(f"combined_df shape: {combined_df.shape}")
print("\n各数据集样本数：")
print(combined_df["dataset"].value_counts())

display(combined_df.head(10))

combined_df shape: (500, 16)

各数据集样本数：
dataset
mt_bench_human_judgments    350
mmlu_pro                    150
Name: count, dtype: int64


,question_id,model_a,model_b,winner,judge,conversation_a,conversation_b,turn,dataset,question,options,answer,answer_index,cot_content,category,src
0,121,gpt-3.5-turbo,gpt-4,model_a,expert_8,[{'content': 'Develop a Python program that re...,[{'content': 'Develop a Python program that re...,2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,112,llama-13b,vicuna-13b-v1.2,model_a,expert_32,[{'content': 'A tech startup invests $8000 in ...,[{'content': 'A tech startup invests $8000 in ...,2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,138,vicuna-13b-v1.2,gpt-3.5-turbo,model_b,expert_42,[{'content': 'Analyze the following customer r...,[{'content': 'Analyze the following customer r...,2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,99,gpt-4,gpt-3.5-turbo,tie,expert_11,[{'content': 'Suppose you are a mathematician ...,[{'content': 'Suppose you are a mathematician ...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,93,vicuna-13b-v1.2,gpt-4,model_b,expert_29,[{'content': 'Imagine yourself as a doctor tas...,[{'content': 'Imagine yourself as a doctor tas...,2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,133,gpt-3.5-turbo,vicuna-13b-v1.2,tie,author_0,[{'content': 'Extract the following informatio...,[{'content': 'Extract the following informatio...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,141,llama-13b,gpt-4,model_b,author_3,"[{'content': 'In the field of quantum physics,...","[{'content': 'In the field of quantum physics,...",2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,108,gpt-3.5-turbo,vicuna-13b-v1.2,tie,expert_53,[{'content': 'Which word does not belong with ...,[{'content': 'Which word does not belong with ...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,94,claude-v1,alpaca-13b,model_a,expert_2,[{'content': 'Please take on the role of a rel...,[{'content': 'Please take on the role of a rel...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,152,gpt-3.5-turbo,llama-13b,model_a,expert_57,[{'content': 'How do the stages of life shape ...,[{'content': 'How do the stages of life shape ...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
combined_df.to_json("src/llmjudgetempcausal/assets/combined_dataset_primitive.jsonl", orient="records", lines=True)

In [20]:
mmlu_df.to_json("src/llmjudgetempcausal/assets/mmlu_pro_primitive.jsonl", orient="records", lines=True)


In [29]:
from openai import OpenAI
from tqdm import tqdm
from jinja2 import Template
from pathlib import Path
import os
import random
import json
import numpy as np
import pandas as pd

SERVICES = {
    "http://0.0.0.0:8000/v1": "Qwen/Qwen2.5-14B-Instruct",
    "http://10.6.32.18:8001/v1": "google/gemma-3-12b-it",
    "http://10.6.32.18:8002/v1": "meta-llama/Llama-3.1-8B-Instruct",
}

API_KEY = os.getenv("VLLM_API_KEY", "token-abc123")

OUTPUT_JSONL = Path("src/llmjudgetempcausal/assets/mmlu_pro_answers_stream.jsonl")
OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

clients = {}
served_model_id = SERVICES.copy()
for base_url in SERVICES:
    clients[base_url] = OpenAI(
        base_url=base_url,
        api_key=API_KEY,
        timeout=120.0,
        max_retries=2,
    )

print("Configured endpoints and models:")
for k, v in served_model_id.items():
    print(f"  {k} -> {v}")

PROMPT_TEMPLATE = Template(
    """
Answer the following multiple-choice question.
Return your final choice and a brief reason.

Question:
{{ question }}

Options:
{% for option in options %}{{ letters[loop.index0] }}. {{ option }}
{% endfor %}
""".strip()
)


def normalize_options(options):
    if isinstance(options, (list, tuple)):
        return list(options)
    return [str(options)]


def build_prompt(row):
    options = normalize_options(row.get("options", []))
    return PROMPT_TEMPLATE.render(
        question=row["question"],
        options=options,
        letters="ABCDEFGHIJKLMNOPQRSTUVWXYZ",
    )


def build_messages(model_id, prompt):
    # Gemma 3 在 vLLM 中要求严格的 user/assistant 轮替，避免 system role 更稳
    if "gemma-3" in model_id.lower():
        return [{"role": "user", "content": prompt}]

    return [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]


def ask_one(base_url, prompt):
    model_id = served_model_id[base_url]
    client = clients[base_url]

    resp = client.chat.completions.create(
        model=model_id,
        messages=build_messages(model_id, prompt),
        temperature=1.0,
    )

    text = (resp.choices[0].message.content or "").strip()
    return model_id, text


def to_json_safe(value):
    if value is None:
        return None

    if isinstance(value, (str, bool)):
        return value

    if isinstance(value, (int, float)):
        if isinstance(value, float) and pd.isna(value):
            return None
        return value

    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()

    if isinstance(value, np.ndarray):
        return [to_json_safe(v) for v in value.tolist()]

    if isinstance(value, (list, tuple)):
        return [to_json_safe(v) for v in value]

    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}

    if isinstance(value, (pd.Timestamp, pd.Timedelta)):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return str(value)


def row_to_jsonable(row_idx):
    record = mmlu_df.loc[row_idx].to_dict()
    clean_record = {k: to_json_safe(v) for k, v in record.items()}
    clean_record["row_idx"] = int(row_idx)
    return clean_record


# 确保目标列存在
for col in ["model_a", "model_b", "conversation_a", "conversation_b"]:
    if col not in mmlu_df.columns:
        mmlu_df[col] = None

# 断点续跑：读取已写入的 row_idx（仅成功行计入 processed）
processed = set()
if OUTPUT_JSONL.exists():
    with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                if "row_idx" in obj and "row_error" not in obj:
                    processed.add(int(obj["row_idx"]))
            except Exception:
                # 跳过损坏行，不中断整体流程
                continue

print(f"Resume mode: 已有 {len(processed)} 条成功行，输出文件: {OUTPUT_JSONL}")

base_urls = list(clients.keys())

for idx, row in tqdm(mmlu_df.iterrows(), total=len(mmlu_df)):
    idx = int(idx)
    if idx in processed:
        continue

    try:
        prompt = build_prompt(row)

        # 对每一行使用固定随机种子，保证断点续跑后仍可复现
        row_rng = random.Random(42 + idx)
        url_a, url_b = row_rng.sample(base_urls, 2)

        try:
            model_a, answer_a = ask_one(url_a, prompt)
        except Exception as e:
            model_a, answer_a = served_model_id[url_a], f"ERROR: {e}"

        try:
            model_b, answer_b = ask_one(url_b, prompt)
        except Exception as e:
            model_b, answer_b = served_model_id[url_b], f"ERROR: {e}"

        mmlu_df.at[idx, "model_a"] = model_a
        mmlu_df.at[idx, "model_b"] = model_b
        mmlu_df.at[idx, "conversation_a"] = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer_a},
        ]
        mmlu_df.at[idx, "conversation_b"] = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer_b},
        ]

        # 一行一行落盘（jsonl）
        record = row_to_jsonable(idx)
        with OUTPUT_JSONL.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

        processed.add(idx)

    except Exception as e:
        # 行级异常也写入 jsonl，后续可继续跑
        error_record = {"row_idx": idx, "row_error": str(e)}
        with OUTPUT_JSONL.open("a", encoding="utf-8") as f:
            f.write(json.dumps(error_record, ensure_ascii=False) + "\n")
        continue

print("\nDone. 预览前 3 条：")
display(mmlu_df[["question", "model_a", "model_b", "conversation_a", "conversation_b"]].head(3))
print(f"\n已写入: {OUTPUT_JSONL}")

Configured endpoints and models:
  http://0.0.0.0:8000/v1 -> Qwen/Qwen2.5-14B-Instruct
  http://10.6.32.18:8001/v1 -> google/gemma-3-12b-it
  http://10.6.32.18:8002/v1 -> meta-llama/Llama-3.1-8B-Instruct
Resume mode: 已有 149 条成功行，输出文件: src/llmjudgetempcausal/assets/mmlu_pro_answers_stream.jsonl


100%|██████████| 150/150 [01:44<00:00,  1.43it/s]


Done. 预览前 3 条：


,question,model_a,model_b,conversation_a,conversation_b
0,This question refers to the following informat...,meta-llama/Llama-3.1-8B-Instruct,Qwen/Qwen2.5-14B-Instruct,"[{'role': 'user', 'content': 'Answer the follo...","[{'role': 'user', 'content': 'Answer the follo..."
1,You are asked to determine the price of a Euro...,Qwen/Qwen2.5-14B-Instruct,google/gemma-3-12b-it,"[{'role': 'user', 'content': 'Answer the follo...","[{'role': 'user', 'content': 'Answer the follo..."
2,When was the major shift by Greek philosopher...,google/gemma-3-12b-it,Qwen/Qwen2.5-14B-Instruct,"[{'role': 'user', 'content': 'Answer the follo...","[{'role': 'user', 'content': 'Answer the follo..."



已写入: src/llmjudgetempcausal/assets/mmlu_pro_answers_stream.jsonl


In [30]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm
import pandas as pd

load_dotenv()  # 读取项目根目录的 .env

JUDGE_MODEL = "gpt-5.4"
ANSWERS_JSONL  = Path("src/llmjudgetempcausal/assets/mmlu_pro_answers_stream.jsonl")
JUDGED_JSONL   = Path("src/llmjudgetempcausal/assets/mmlu_pro_judged_stream.jsonl")
JUDGED_JSONL.parent.mkdir(parents=True, exist_ok=True)

judge_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

raw_rows = []
with ANSWERS_JSONL.open("r", encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        obj = json.loads(line)
        if "row_error" not in obj:
            raw_rows.append(obj)

answers_df = pd.DataFrame(raw_rows)
print(f"Loaded {len(answers_df)} rows from {ANSWERS_JSONL}")
display(answers_df[["row_idx", "question", "model_a", "model_b"]].head(3))


# ── Judge prompt builder ───────────────────────────────────────────────────
JUDGE_SYSTEM = (
    "You are an impartial judge evaluating answers to multiple-choice questions. "
    "Your task is to decide which answer is better or whether they are equally good.\n"
    "Respond ONLY with a JSON object — no markdown, no extra text — in this exact format:\n"
    '{"winner": "<A|B|tie>", "reason": "<one concise sentence>"}'
)

def build_judge_prompt(row: dict) -> str:
    options = row.get("options") or []
    if isinstance(options, str):
        try:
            options = json.loads(options)
        except Exception:
            options = [options]

    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    options_text = "\n".join(
        f"{letters[i]}. {opt}" for i, opt in enumerate(options)
    )

    answer_a = ""
    answer_b = ""
    for turn in (row.get("conversation_a") or []):
        if turn.get("role") == "assistant":
            answer_a = turn.get("content", "")
    for turn in (row.get("conversation_b") or []):
        if turn.get("role") == "assistant":
            answer_b = turn.get("content", "")

    correct = row.get("answer", "")

    return (
        f"## Question\n{row['question']}\n\n"
        f"## Options\n{options_text}\n\n"
        f"## Correct Answer\n{correct}\n\n"
        f"## Answer A  (model: {row.get('model_a', 'unknown')})\n{answer_a}\n\n"
        f"## Answer B  (model: {row.get('model_b', 'unknown')})\n{answer_b}\n\n"
        "Which answer is better? Reply with JSON only."
    )


def call_judge(judge_prompt: str) -> dict:
    resp = judge_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user",   "content": judge_prompt},
        ],
        temperature=0.0,
    )
    raw = (resp.choices[0].message.content or "").strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    return json.loads(raw)


# ── 断点续跑 ───────────────────────────────────────────────────────────────
judged_set = set()
if JUDGED_JSONL.exists():
    with JUDGED_JSONL.open("r", encoding="utf-8") as fh:
        for line in fh:
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                if "row_idx" in obj and "judge_error" not in obj:
                    judged_set.add(int(obj["row_idx"]))
            except Exception:
                continue

print(f"Resume: {len(judged_set)} rows already judged.")

# ── Main loop ──────────────────────────────────────────────────────────────
for _, row in tqdm(answers_df.iterrows(), total=len(answers_df)):
    row_idx = int(row["row_idx"])
    if row_idx in judged_set:
        continue

    try:
        judge_prompt = build_judge_prompt(row.to_dict())
        result = call_judge(judge_prompt)

        raw_winner = result.get("winner", "").strip().upper()
        if raw_winner == "A":
            winner = "model_a"
        elif raw_winner == "B":
            winner = "model_b"
        else:
            winner = "tie"

        # 保留原始所有列，再追加 judge 字段
        record = row.to_dict()
        record.update({
            "winner":       winner,
            "judge_model":  JUDGE_MODEL,
            "judge_prompt": judge_prompt,
            "judge_reason": result.get("reason", ""),
        })

    except Exception as e:
        record = {
            "row_idx":     row_idx,
            "judge_error": str(e),
        }

    with JUDGED_JSONL.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

    if "judge_error" not in record:
        judged_set.add(row_idx)

# ── 加载结果 ───────────────────────────────────────────────────────────────
judged_rows = []
with JUDGED_JSONL.open("r", encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        obj = json.loads(line)
        if "judge_error" not in obj:
            judged_rows.append(obj)

judged_df = pd.DataFrame(judged_rows)
print(f"\nDone. {len(judged_df)} rows judged.")
print(judged_df["winner"].value_counts())
display(judged_df[["row_idx", "question", "model_a", "model_b", "winner", "judge_model", "judge_reason"]].head(5))
print(f"\n已写入: {JUDGED_JSONL}")

Loaded 150 rows from src/llmjudgetempcausal/assets/mmlu_pro_answers_stream.jsonl


,row_idx,question,model_a,model_b
0,0,This question refers to the following informat...,meta-llama/Llama-3.1-8B-Instruct,Qwen/Qwen2.5-14B-Instruct
1,1,You are asked to determine the price of a Euro...,Qwen/Qwen2.5-14B-Instruct,google/gemma-3-12b-it
2,2,When was the major shift by Greek philosopher...,google/gemma-3-12b-it,Qwen/Qwen2.5-14B-Instruct


Resume: 0 rows already judged.


100%|██████████| 150/150 [03:17<00:00,  1.32s/it]


Done. 150 rows judged.
winner
model_a    53
model_b    51
tie        46
Name: count, dtype: int64


,row_idx,question,model_a,model_b,winner,judge_model,judge_reason
0,0,This question refers to the following informat...,meta-llama/Llama-3.1-8B-Instruct,Qwen/Qwen2.5-14B-Instruct,model_b,gpt-5.4,B correctly selects F and explains Logan’s gri...
1,1,You are asked to determine the price of a Euro...,Qwen/Qwen2.5-14B-Instruct,google/gemma-3-12b-it,model_a,gpt-5.4,A correctly incorporates the dividend yield δ ...
2,2,When was the major shift by Greek philosopher...,google/gemma-3-12b-it,Qwen/Qwen2.5-14B-Instruct,tie,gpt-5.4,Both answers choose the same incorrect option ...
3,3,Which of the following criticisms of Llewellyn...,google/gemma-3-12b-it,meta-llama/Llama-3.1-8B-Instruct,model_a,gpt-5.4,A selects the correct option C and gives a con...
4,4,Is the recognition of foreign judgments subjec...,Qwen/Qwen2.5-14B-Instruct,google/gemma-3-12b-it,model_a,gpt-5.4,A is slightly better because it directly contr...



已写入: src/llmjudgetempcausal/assets/mmlu_pro_judged_stream.jsonl


In [32]:
# 给两个 df 分别加上来源标记
mt_df = sampled_df.copy()
mt_df["dataset"] = "mt_bench_human_judgments"

judged_df = judged_df.copy()
judged_df["dataset"] = "mmlu_pro"

# 垂直拼接（保留所有列，缺失列自动补 NaN）
combined_df = pd.concat([mt_df, judged_df], ignore_index=True, sort=False)

print(f"combined_df shape: {combined_df.shape}")
print("\n各数据集样本数：")
print(combined_df["dataset"].value_counts())

display(combined_df.head(10))
combined_df.to_json("src/llmjudgetempcausal/assets/combined_dataset_judged.jsonl", orient="records", lines=True)

combined_df shape: (500, 20)

各数据集样本数：
dataset
mt_bench_human_judgments    350
mmlu_pro                    150
Name: count, dtype: int64


,question_id,model_a,model_b,winner,judge,conversation_a,conversation_b,turn,dataset,question,options,answer,answer_index,cot_content,category,src,row_idx,judge_model,judge_prompt,judge_reason
0,121,gpt-3.5-turbo,gpt-4,model_a,expert_8,[{'content': 'Develop a Python program that re...,[{'content': 'Develop a Python program that re...,2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,112,llama-13b,vicuna-13b-v1.2,model_a,expert_32,[{'content': 'A tech startup invests $8000 in ...,[{'content': 'A tech startup invests $8000 in ...,2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,138,vicuna-13b-v1.2,gpt-3.5-turbo,model_b,expert_42,[{'content': 'Analyze the following customer r...,[{'content': 'Analyze the following customer r...,2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,99,gpt-4,gpt-3.5-turbo,tie,expert_11,[{'content': 'Suppose you are a mathematician ...,[{'content': 'Suppose you are a mathematician ...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,93,vicuna-13b-v1.2,gpt-4,model_b,expert_29,[{'content': 'Imagine yourself as a doctor tas...,[{'content': 'Imagine yourself as a doctor tas...,2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,133,gpt-3.5-turbo,vicuna-13b-v1.2,tie,author_0,[{'content': 'Extract the following informatio...,[{'content': 'Extract the following informatio...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,141,llama-13b,gpt-4,model_b,author_3,"[{'content': 'In the field of quantum physics,...","[{'content': 'In the field of quantum physics,...",2.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,108,gpt-3.5-turbo,vicuna-13b-v1.2,tie,expert_53,[{'content': 'Which word does not belong with ...,[{'content': 'Which word does not belong with ...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,94,claude-v1,alpaca-13b,model_a,expert_2,[{'content': 'Please take on the role of a rel...,[{'content': 'Please take on the role of a rel...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,152,gpt-3.5-turbo,llama-13b,model_a,expert_57,[{'content': 'How do the stages of life shape ...,[{'content': 'How do the stages of life shape ...,1.0,mt_bench_human_judgments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
combined_df.dataset.value_counts()

dataset
mt_bench_human_judgments    350
mmlu_pro                    150
Name: count, dtype: int64

In [36]:
combined_df[combined_df["dataset"] == "mmlu_pro"].iloc[0]

question_id                                                    4715
model_a                            meta-llama/Llama-3.1-8B-Instruct
model_b                                   Qwen/Qwen2.5-14B-Instruct
winner                                                      model_b
judge                                                           NaN
conversation_a    [{'role': 'user', 'content': 'Answer the follo...
conversation_b    [{'role': 'user', 'content': 'Answer the follo...
turn                                                            NaN
dataset                                                    mmlu_pro
question          This question refers to the following informat...
options           [Logan believes that Indians need to find stre...
answer                                                            F
answer_index                                                    5.0
cot_content                                                        
category                                        

In [37]:
combined_df[combined_df["dataset"] == "mt_bench_human_judgments"].iloc[0]

question_id                                                     121
model_a                                               gpt-3.5-turbo
model_b                                                       gpt-4
winner                                                      model_a
judge                                                      expert_8
conversation_a    [{'content': 'Develop a Python program that re...
conversation_b    [{'content': 'Develop a Python program that re...
turn                                                            2.0
dataset                                    mt_bench_human_judgments
question                                                        NaN
options                                                         NaN
answer                                                          NaN
answer_index                                                    NaN
cot_content                                                     NaN
category                                        

In [ ]:
combined_df["reference_answer"] = combined_df["answer"]

In [50]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

load_dotenv()

REFERENCE_MODEL = "gpt-5-mini"
REFERENCE_OUTPUT_JSONL = Path("src/llmjudgetempcausal/assets/combined_reference_answers_stream.jsonl")
REFERENCE_FINAL_JSONL = Path("src/llmjudgetempcausal/assets/combined_dataset_with_reference.jsonl")
REFERENCE_OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

reference_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

REFERENCE_SYSTEM = (
    "You are an expert assistant writing high-quality reference answers. "
    "Be accurate, concise, and helpful."
)


def _to_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        try:
            parsed = json.loads(s)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []


def build_mt_reference_prompt(row):
    conv_a = _to_list(row.get("conversation_a"))
    user_turns = [
        m.get("content", "")
        for m in conv_a
        if isinstance(m, dict) and m.get("role") == "user"
    ]

    if not user_turns and isinstance(row.get("question"), str):
        user_turns = [row["question"]]
    if not user_turns and isinstance(row.get("question_text"), str):
        user_turns = [row["question_text"]]

    turn_raw = row.get("turn", 1)
    try:
        turn = max(1, int(turn_raw))
    except Exception:
        turn = 1

    if user_turns:
        target_idx = min(turn - 1, len(user_turns) - 1)
        target_question = user_turns[target_idx]
        history_turns = user_turns[:target_idx]
    else:
        target_question = ""
        history_turns = []

    history_text = "\n".join([f"- {q}" for q in history_turns]) if history_turns else "None"

    return (
        "Task: Write a gold reference answer for this MT-Bench user query.\n"
        "Requirements:\n"
        "1) Directly answer the current user query.\n"
        "2) Be factually accurate and reasonably concise.\n"
        "3) If prior user turns exist, keep the answer consistent with that context.\n\n"
        f"Previous user turns:\n{history_text}\n\n"
        f"Current user query:\n{target_question}\n\n"
        "Output only the reference answer text."
    )


def build_mmlu_reference_prompt(row):
    question = str(row.get("question", ""))
    options = _to_list(row.get("options"))
    answer = str(row.get("answer", ""))

    answer_index = row.get("answer_index", None)
    correct_option_text = ""
    correct_label = ""

    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    try:
        if pd.notna(answer_index):
            idx = int(answer_index)
            if 0 <= idx < len(options):
                correct_label = letters[idx]
                correct_option_text = str(options[idx])
    except Exception:
        pass

    if not correct_option_text and options:
        ans_upper = answer.strip().upper()
        if len(ans_upper) == 1 and ans_upper in letters:
            idx = letters.index(ans_upper)
            if idx < len(options):
                correct_label = ans_upper
                correct_option_text = str(options[idx])

    option_lines = "\n".join(
        [f"{letters[i]}. {opt}" for i, opt in enumerate(options)]
    ) if options else "None"

    return (
        "Task: Write an ideal answer with analysis for the following multiple-choice question.\n"
        "You are given the gold answer, and you should produce a high-quality explanation that leads to that answer.\n"
        "Do not mention that you were given the gold answer.\n\n"
        f"Question:\n{question}\n\n"
        f"Options:\n{option_lines}\n\n"
        f"Gold answer label: {correct_label or answer}\n"
        f"Gold answer content: {correct_option_text or answer}\n\n"
        "Output format:\n"
        "Final Answer: <one line>\n"
        "Analysis: <concise reasoning>"
    )


def call_reference_model(prompt):
    resp = reference_client.chat.completions.create(
        model=REFERENCE_MODEL,
        messages=[
            {"role": "system", "content": REFERENCE_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        # temperature=0.2,
    )
    return (resp.choices[0].message.content or "").strip()


# 确保列存在
for col in ["reference_answer", "reference_model", "reference_prompt", "reference_error"]:
    if col not in combined_df.columns:
        combined_df[col] = None


# 断点续跑：从流式文件恢复已成功行
processed = set()
if REFERENCE_OUTPUT_JSONL.exists():
    with REFERENCE_OUTPUT_JSONL.open("r", encoding="utf-8") as fh:
        for line in fh:
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                row_idx = int(obj["row_idx"])
                if "reference_error" not in obj:
                    combined_df.at[row_idx, "reference_answer"] = obj.get("reference_answer")
                    combined_df.at[row_idx, "reference_model"] = obj.get("reference_model")
                    combined_df.at[row_idx, "reference_prompt"] = obj.get("reference_prompt")
                    combined_df.at[row_idx, "reference_error"] = None
                    processed.add(row_idx)
            except Exception:
                continue

print(f"Resume mode: 已有 {len(processed)} 条 reference 已完成")

for idx, row in tqdm(combined_df.iterrows(), total=len(combined_df)):
    idx = int(idx)
    if idx in processed:
        continue

    dataset_name = row.get("dataset", "")

    try:
        if dataset_name == "mt_bench_human_judgments":
            reference_prompt = build_mt_reference_prompt(row)
        elif dataset_name == "mmlu_pro":
            reference_prompt = build_mmlu_reference_prompt(row)
        else:
            reference_prompt = ""

        if not reference_prompt.strip():
            raise ValueError(f"Unsupported or empty prompt for dataset={dataset_name}")

        reference_answer = call_reference_model(reference_prompt)

        combined_df.at[idx, "reference_answer"] = reference_answer
        combined_df.at[idx, "reference_model"] = REFERENCE_MODEL
        combined_df.at[idx, "reference_prompt"] = reference_prompt
        combined_df.at[idx, "reference_error"] = None

        out = {
            "row_idx": idx,
            "dataset": dataset_name,
            "reference_answer": reference_answer,
            "reference_model": REFERENCE_MODEL,
            "reference_prompt": reference_prompt,
        }
        with REFERENCE_OUTPUT_JSONL.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(out, ensure_ascii=False, default=str) + "\n")

        processed.add(idx)

    except Exception as e:
        combined_df.at[idx, "reference_error"] = str(e)

        out = {
            "row_idx": idx,
            "dataset": dataset_name,
            "reference_error": str(e),
        }
        with REFERENCE_OUTPUT_JSONL.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(out, ensure_ascii=False, default=str) + "\n")

# 全量保存
combined_df.to_json(
    REFERENCE_FINAL_JSONL,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("\nDone.")
print(f"Reference stream: {REFERENCE_OUTPUT_JSONL}")
print(f"Final dataset:    {REFERENCE_FINAL_JSONL}")
print(combined_df["dataset"].value_counts())
display(combined_df[["dataset", "reference_model", "reference_error"]].head(10))

Resume mode: 已有 34 条 reference 已完成


100%|██████████| 500/500 [1:55:28<00:00, 13.86s/it]  


Done.
Reference stream: src/llmjudgetempcausal/assets/combined_reference_answers_stream.jsonl
Final dataset:    src/llmjudgetempcausal/assets/combined_dataset_with_reference.jsonl
dataset
mt_bench_human_judgments    350
mmlu_pro                    150
Name: count, dtype: int64


,dataset,reference_model,reference_error
0,mt_bench_human_judgments,gpt-5.4,None
1,mt_bench_human_judgments,gpt-5.4,None
2,mt_bench_human_judgments,gpt-5.4,None
3,mt_bench_human_judgments,gpt-5.4,None
4,mt_bench_human_judgments,gpt-5.4,None
5,mt_bench_human_judgments,gpt-5.4,None
6,mt_bench_human_judgments,gpt-5.4,None
7,mt_bench_human_judgments,gpt-5.4,None
8,mt_bench_human_judgments,gpt-5.4,None
9,mt_bench_human_judgments,gpt-5.4,None
